In [ ]:
# Função para carregar os arquivos
base_path = r"C:\Dumps\todos_enms"

def load_parquet(pattern, columns=None):
    import glob, pandas as pd
    files = glob.glob(pattern)
    if not files:
        return pd.DataFrame()
    df = pd.concat((pd.read_parquet(f, columns=columns) for f in files),
                   ignore_index=True)
    try:
        df = df.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        pass
    return df


In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\GNBCUUPFunction_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
GNBCUUPFunction= df1.copy()

GNBCUUPFunction = GNBCUUPFunction.loc[
    GNBCUUPFunction['endcDataUsageReportEnabled'] == False, 
    ['UF', 'eNB', 'gNBId', 'endcDataUsageReportEnabled']
]

In [ ]:
# df1.to_excel('C:/Camilao/GNBCUUPFunction_endcDataUsageReportEnabled.xlsx', index=None)

In [ ]:
import pandas as pd
from glob import glob

ENBFUNC= pd.read_csv('C:/Dumps/LTE_total/ENodeBFunction.txt',sep='\t',low_memory=False, usecols=['UF', 'eNB', 'endcDataUsageReportEnabled'])
ENBFUNC= ENBFUNC[ENBFUNC['endcDataUsageReportEnabled']==False]

In [ ]:
import pandas as pd

from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

arquivo_txt= 'C:/Camilo/' + '5G_GUtranCellRelation'+ '_'+str(semana_atual)+'.txt'

df= pd.read_csv(arquivo_txt, sep='\t', low_memory=False)
pd.set_option('display.max_columns',None)
ofensores= df[df['isEndcAllowed']== False]


ofensores= ofensores.loc[(~ofensores['nRCellCUId'].str[0:2].isin(['5R', '5U',  '5Z'])),:]



In [ ]:
ofensores1= ofensores[['Regional', 'UF', 'eNB','EUtranCell', 'Neighbor', 'nRCellCUId', 'isEndcAllowed', 'ExternalGNodeBFunction', 'ExternalGUtranCell']]
isEndcAllowed= ofensores1.copy()
isEndcAllowed.dropna(subset= ['Neighbor'],inplace=True)

In [ ]:
import os
import pandas as pd

# Caminho para salvar o arquivo
save_path = "C:/Vivo Reports/5G/5G Mobilidade/"
file_name = "Audit_PSCell_Endc.xlsx"
full_path = os.path.join(save_path, file_name)

# Verificar se o diretório existe, caso contrário, criar
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Salvar os DataFrames no mesmo arquivo Excel usando openpyxl
with pd.ExcelWriter(full_path, engine='openpyxl') as writer:
    GNBCUUPFunction.to_excel(writer, sheet_name='GNBCUUPFunction', index=False)
    ENBFUNC.to_excel(writer, sheet_name='ENBFUNC', index=False)
    isEndcAllowed.to_excel(writer, sheet_name='isEndcAllowed', index=False)

print('Done!!!')